# Import Library

In [1]:
import pandas as pd
import numpy as np
import re
import os
import difflib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import BaseEstimator, TransformerMixin

# Library Khusus Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# MLflow
import mlflow
import mlflow.sklearn
import dagshub

d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Import Data

In [2]:
file_path = "../../data/synthetic_report_dataset.csv" 

# 1. Intip file mentah untuk deteksi pemisah
with open(file_path, 'r', encoding='utf-8') as f:
    baris_pertama = f.readline()
    print("Isi baris pertama file mentah:\n👉", baris_pertama)

pemisah = '|' if '|' in baris_pertama else ','
print(f"Sistem mendeteksi pemisah kolom: '{pemisah}'\n")

# 2. Load Data
nama_kolom = ['teks_keluhan_awam', 'teks_laporan_teknisi', 'kategori_aset', 'severity', 'root_cause', 'tindakan']
df = pd.read_csv(file_path, sep=pemisah, names=nama_kolom, on_bad_lines='skip')

print(f"Baris sebelum dibersihkan: {df.shape[0]}")

# 3. PEMBERSIHAN & PENYELAMATAN DATA
kolom_target = ['kategori_aset', 'severity', 'root_cause', 'tindakan']
baris_patah = df[df['kategori_aset'].isna()].index

for i in baris_patah:
    if i + 1 in df.index and not pd.isna(df.loc[i+1, 'kategori_aset']):
        teks_atas = str(df.loc[i, 'teks_keluhan_awam']).strip()
        teks_bawah = str(df.loc[i+1, 'teks_keluhan_awam']).strip()
        if teks_atas != 'nan':
            df.loc[i+1, 'teks_keluhan_awam'] = teks_atas + " " + teks_bawah

df = df.dropna(subset=kolom_target)
df = df[df['teks_keluhan_awam'].astype(str).str.contains('teks_keluhan', case=False) == False]
df = df.reset_index(drop=True)

df['input_teks'] = df['teks_keluhan_awam'].astype(str) + " " + df['teks_laporan_teknisi'].astype(str)

print(f"Total data siap diproses: {df.shape[0]}")

Isi baris pertama file mentah:
👉 AC di ruang meeting lantai 2 netes air terus nih,"Saluran pembuangan air kondensasi tersumbat lumut dan debu, sudah dibersihkan tuntas",HVAC,Sedang,Tersumbat,Pembersihan

Sistem mendeteksi pemisah kolom: ','

Baris sebelum dibersihkan: 2797
Total data siap diproses: 2796


# Cleaning & Preprocessing

In [3]:
print("=== Standardisasi Label Target (Y) ===")

# List valid (HARUS lowercase untuk pencocokan)
valid_severity_lower = ['tinggi', 'sedang', 'rendah']

def robust_label_encoder(text, valid_list_lower):
    if pd.isna(text):
        return "UNKNOWN"
        
    text_clean = str(text).strip().lower()
    
    # 1. Exact Match
    if text_clean in valid_list_lower:
        return text_clean.capitalize() 
        
    # 2. Fuzzy Match (dengan cutoff ketat agar tidak asal tebak)
    closest_match = difflib.get_close_matches(text_clean, valid_list_lower, n=1, cutoff=0.5)
    
    if closest_match:
        return closest_match[0].capitalize()
    else:
        return "UNKNOWN" # Jangan biarkan typo lolos!

# Terapkan ke kolom target
df['severity'] = df['severity'].apply(lambda x: robust_label_encoder(x, valid_severity_lower))

# Buang baris yang labelnya benar-benar tidak bisa diselamatkan (UNKNOWN)
df = df[df['severity'] != 'UNKNOWN']

Y = df[kolom_target]

print("\nDistribusi Label Severity SEKARANG (Sudah Bersih):")
print(Y['severity'].value_counts())
print("\nNilai Unik:", Y['severity'].unique())

=== Standardisasi Label Target (Y) ===

Distribusi Label Severity SEKARANG (Sudah Bersih):
severity
Tinggi    1173
Sedang    1012
Rendah     611
Name: count, dtype: int64

Nilai Unik: ['Sedang' 'Tinggi' 'Rendah']


In [4]:
print("Menyiapkan Sastrawi Preprocessor (Production Ready)...")

# Membungkus Sastrawi ke dalam Class Scikit-Learn agar bisa di-save ke dalam Pipeline
class SastrawiPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.stemmer = StemmerFactory().create_stemmer()
        self.stopword_remover = StopWordRemoverFactory().create_stop_word_remover()
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        print("Membersihkan teks...")
        return X.apply(self._clean_text)
        
    def _clean_text(self, text):
        text = str(text).lower()
        text = re.sub(r'[^a-z\s]', '', text)
        text = self.stopword_remover.remove(text)
        text = self.stemmer.stem(text)
        return text

# Inisialisasi Preprocessor
preprocessor = SastrawiPreprocessor()

# Kita bersihkan data sekarang untuk keperluan training
df['clean_teks'] = preprocessor.transform(df['input_teks'])
print("Data Cleaning Selesai!")

Menyiapkan Sastrawi Preprocessor (Production Ready)...
Membersihkan teks...
Data Cleaning Selesai!


In [5]:
import pandas as pd

# Kita asumsikan 'kolom_target' sudah didefinisikan sebelumnya:
# kolom_target = ['kategori_aset', 'severity', 'root_cause', 'tindakan']

# Looping untuk setiap kolom di dalam DataFrame Y
for col in kolom_target:
    print(f"=======================================================")
    print(f"                 KOLOM: {col.upper()}")
    print(f"=======================================================\n")
    
    # 1. Cek Distribusi / Imbalance Kelas
    print(f"=== Distribusi Label di Kolom {col} ===")
    print(Y[col].value_counts(dropna=False)) 
    
    print("\n-------------------------------------------------\n")
    
    # 2. Cek Typo: Melihat semua nilai unik yang pernah diketik
    print(f"=== Nilai Unik di Kolom {col} ===")
    print(Y[col].unique())
    print("\n\n") # Memberikan jarak antar kolom

                 KOLOM: KATEGORI_ASET

=== Distribusi Label di Kolom kategori_aset ===
kategori_aset
Mesin Produksi    715
HVAC              709
Kelistrikan       697
Pompa Air         675
Name: count, dtype: int64

-------------------------------------------------

=== Nilai Unik di Kolom kategori_aset ===
['HVAC' 'Pompa Air' 'Mesin Produksi' 'Kelistrikan']



                 KOLOM: SEVERITY

=== Distribusi Label di Kolom severity ===
severity
Tinggi    1173
Sedang    1012
Rendah     611
Name: count, dtype: int64

-------------------------------------------------

=== Nilai Unik di Kolom severity ===
['Sedang' 'Tinggi' 'Rendah']



                 KOLOM: ROOT_CAUSE

=== Distribusi Label di Kolom root_cause ===
root_cause
Keausan       663
Overheat      601
Usia Pakai    561
Tersumbat     509
Konsleting    462
Name: count, dtype: int64

-------------------------------------------------

=== Nilai Unik di Kolom root_cause ===
['Tersumbat' 'Keausan' 'Konsleting' 'Overheat' 'Usia Pakai'

# Train-Test Split

In [6]:
# Pemisahan fitur dan target
X = df['clean_teks']
y = df[kolom_target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Jumlah Data Latih: {X_train.shape[0]}, Data Uji: {X_test.shape[0]}")

Jumlah Data Latih: 2236, Data Uji: 560


# DATA AUGMENTATION

In [8]:
# ==========================================
# CELL 6: DATA AUGMENTATION (MENGGUNAKAN GPU/CUDA)
# ==========================================
import nlpaug.augmenter.word as naw
import pandas as pd
from tqdm import tqdm
import torch

# 1. Cek apakah GPU/CUDA benar-benar tersedia
if torch.cuda.is_available():
    device = 'cuda'
    print(f"🚀 Mantap! Menggunakan GPU: {torch.cuda.get_device_name(0)}")
else:
    device = 'cpu'
    print("⚠️ GPU tidak ditemukan, menggunakan CPU.")

print("\n1. Menyiapkan Library NLP Augmentation (IndoBERT)...")
# Menggunakan IndoBERT dengan parameter device untuk menembak ke GPU
aug = naw.ContextualWordEmbsAug(
    model_path='indobenchmark/indobert-base-p1', 
    action="substitute",
    aug_p=0.2,
    device=device # <--- INI KUNCI UTAMANYA
)

def library_augmentation(text):
    try:
        augmented_text = aug.augment(text)[0]
        return augmented_text
    except Exception as e:
        return text

print("2. Mulai proses augmentasi data latih...")
tqdm.pandas(desc="Augmenting Data")
X_train_aug = X_train.progress_apply(library_augmentation)

# Gabungkan data asli dengan data hasil augmentasi
X_train_final = pd.concat([X_train, X_train_aug])
y_train_final = pd.concat([y_train, y_train])

print(f"\n✅ Selesai! Data Latih setelah Augmentasi: {X_train_final.shape[0]} baris.")

🚀 Mantap! Menggunakan GPU: NVIDIA GeForce RTX 3060 Laptop GPU

1. Menyiapkan Library NLP Augmentation (IndoBERT)...


d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\transformers\modeling_utils.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer b

2. Mulai proses augmentasi data latih...


Augmenting Data: 100%|██████████| 2236/2236 [05:31<00:00,  6.74it/s]


✅ Selesai! Data Latih setelah Augmentasi: 4472 baris.


# FEATURE ENGINEERING & MODELING

In [9]:
# 1. Setup MLflow & DagsHub
dagshub.init(repo_owner='NazeeraAlthea', repo_name='exigen-smart-maintenance', mlflow=True)
mlflow.set_experiment("Smart_Ticketing_Baseline_TFIDF")

with mlflow.start_run(run_name="TFIDF_RF_Production_Pipeline"):
    
    print("Mulai Hyperparameter Tuning dengan GridSearchCV dan MLflow Tracking...")
    
    # Perhatikan: Kita hanya memasukkan TF-IDF dan Classifier di pipeline training ini
    # karena teks X_train_final sudah di-clean & di-augmentasi.
    # Nanti di backend (production), pipeline ini akan digabung dengan SastrawiPreprocessor.
    pipeline_training = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', MultiOutputClassifier(RandomForestClassifier(random_state=42)))
    ])

    param_grid = {
        'tfidf__max_features': [1000, 2000],
        'clf__estimator__n_estimators': [100, 200],
        'clf__estimator__max_depth': [None, 10, 20]
    }

    grid_search = GridSearchCV(pipeline_training, param_grid, cv=3, n_jobs=1, verbose=1)
    grid_search.fit(X_train_final, y_train_final)

    print(f"\nBest Parameters: {grid_search.best_params_}")
    mlflow.log_params(grid_search.best_params_)
    
    # 2. Evaluasi Final
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print("\n=== HASIL EVALUASI FINAL ===")
    exact_match_manual = (y_test.values == y_pred).all(axis=1).mean()
    mlflow.log_metric("exact_match_ratio", exact_match_manual)
    print(f"Exact Match Ratio (Benar Semua 4 Kolom): {exact_match_manual * 100:.2f}%")

    print("\n--- Akurasi Individu per Target ---")
    for i, col in enumerate(kolom_target):
        acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
        mlflow.log_metric(f"accuracy_{col}", acc)
        print(f"Akurasi {col:15}: {acc * 100:.2f}%")
        
    # 3. Log Model ke MLflow
    mlflow.sklearn.log_model(best_model, "best_rf_tfidf_model")
    print("\n✅ Run MLflow Selesai! Model dan metrik berhasil dikirim ke DagsHub.")

Accessing as NazeeraAlthea

Initialized MLflow to track repo "NazeeraAlthea/exigen-smart-maintenance"

Repository NazeeraAlthea/exigen-smart-maintenance initialized!

Mulai Hyperparameter Tuning dengan GridSearchCV dan MLflow Tracking...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best Parameters: {'clf__estimator__max_depth': None, 'clf__estimator__n_estimators': 200, 'tfidf__max_features': 1000}

=== HASIL EVALUASI FINAL ===
Exact Match Ratio (Benar Semua 4 Kolom): 77.14%

--- Akurasi Individu per Target ---
Akurasi kategori_aset  : 100.00%
Akurasi severity       : 77.86%
Akurasi root_cause     : 99.29%


2026/05/12 12:30:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Akurasi tindakan       : 99.11%


2026/05/12 12:30:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Run MLflow Selesai! Model dan metrik berhasil dikirim ke DagsHub.
🏃 View run TFIDF_RF_Production_Pipeline at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/1/runs/cb3a6dacf07f4f7eb123f2c95fe11fc1
🧪 View experiment at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/1
